Um jeito de econimizar VRAM

# 1. Normal e DDP

Args:
- function = module do pytorch
- *agrs = Inputs
- use_reetrant = 

In [ ]:
from torch.utils.checkpoint import checkpoint
import torch.nn as nn
import torch

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(2, 2)
        self.l2 = nn.Linear(2, 2)

    def forward(self, x):

        x = checkpoint(self.l1, x, use_reentrant=False)
        x = checkpoint(self.l2, x, use_reentrant=False)
        return x

# 2. FSDP



1. checkpooint_wrapper = O que as camadas devem e como fazer

    - AGRS:
    1. - module 
    2. - checkpoint_impl = Como vai organizar a reeorganização (CheckpointImpl.NO_REETRANT or CheckpointImpl.REETRANT)
    3. - offload_to_cpu = Tenta mover as camadas para a CPU ao inves de descartar

2. CheckpointImpl = Como vai organizar as coisas (.NO_REETRANT or .REETRANT)

3. apply_activation_checkpointing_wrapper = Função que aplica o checkpoint no model (Need to be applied after the FSDP function)

    - ARGS OF THE APPLY:
    1. - check_fn = Somente as camadas escolhidas sofrem isso
    2. - checkpooint_wrapper_fn = O que as camadas escolhidas devem fazer

In [ ]:
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.algorithms._checkpoint.checkpoint_wrapper import (
    checkpoint_wrapper,
    CheckpointImpl,
    apply_activation_checkpointing_wrapper
)

def check_fn(submodule):
    return isinstance(submodule, TransformerBlock) # -> Will return True or False

checkpoint_wrapper_fn = lambda x: checkpoint_wrapper(
    x, 
    checkpoint_impl= CheckpointImpl.NO_REENTRANT,
    offload_to_cpu = False
)

# Instace the model
model = None

# Now you do the FSDP
model = FSDP(model, ...)
apply_activation_checkpointing_wrapper( # MAIN FUNCTION !!!!! ----- !!!!!!!! After the FSDP
    model,
    checkpoint_wrapper_fn=checkpoint_wrapper_fn,
    check_fn=check_fn
)